# 雷达反射率时间插值到整 6 分钟

目标：读取 `test/Z9002_*.npy` 雷达反射率数据，根据文件名中的时间戳做时间线性插值，并输出到整 6 分钟时刻。

例如目标时刻会落在：`00:00, 00:06, 00:12, ..., 04:48, 04:54, 05:00`。

注意：本 notebook 默认只在相邻雷达帧时间差不超过 `MAX_GAP_MINUTES` 时插值，避免跨越长时间缺测段强行插值。

In [ ]:
from bisect import bisect_right
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

TEST_DIR = Path('test')
OUTPUT_DIR = Path('test_radar_6min')

# 只在相邻雷达帧间隔不超过该阈值时做插值。
# 当前数据正常间隔约 5-6 分钟，少量约 10 分钟，所以 30 分钟比较保守。
MAX_GAP_MINUTES = 30

OUTPUT_DIR.mkdir(exist_ok=True)
assert TEST_DIR.exists(), f'输入目录不存在: {TEST_DIR.resolve()}'

TEST_DIR.resolve(), OUTPUT_DIR.resolve()

## 1. 解析雷达文件时间

In [ ]:
def parse_radar_time(path: Path) -> datetime:
    """从 Z9002_YYYYmmddHHMMSS.npy 文件名中解析时间。"""
    timestamp = path.stem.split('_', 1)[1]
    return datetime.strptime(timestamp, '%Y%m%d%H%M%S')


radar_files = sorted(TEST_DIR.glob('Z9002_*.npy'), key=parse_radar_time)
radar_times = [parse_radar_time(path) for path in radar_files]

radar_index = pd.DataFrame(
    {
        'time': radar_times,
        'file': [path.name for path in radar_files],
    }
)

print('雷达文件数量:', len(radar_files))
print('起始时间:', radar_times[0])
print('结束时间:', radar_times[-1])
radar_index.head(), radar_index.tail()

## 2. 按时间缺口切分连续片段

如果两个相邻雷达文件间隔太大，中间不做插值。这样可以避免从 `2024-05-27` 直接插到 `2024-06-18`。

In [ ]:
def split_segments(times, max_gap_minutes=30):
    """把时间序列按最大允许间隔切成多个连续片段，返回 (start_idx, end_idx) 闭区间。"""
    if not times:
        return []

    max_gap = timedelta(minutes=max_gap_minutes)
    segments = []
    start = 0
    for i in range(1, len(times)):
        if times[i] - times[i - 1] > max_gap:
            segments.append((start, i - 1))
            start = i
    segments.append((start, len(times) - 1))
    return segments


segments = split_segments(radar_times, MAX_GAP_MINUTES)
segment_summary = pd.DataFrame(
    [
        {
            'segment': idx,
            'start': radar_times[s],
            'end': radar_times[e],
            'source_frames': e - s + 1,
        }
        for idx, (s, e) in enumerate(segments)
    ]
)

segment_summary

## 3. 生成整 6 分钟目标时刻

In [ ]:
def floor_to_6min(t: datetime) -> datetime:
    minute = (t.minute // 6) * 6
    return t.replace(minute=minute, second=0, microsecond=0)


def ceil_to_6min(t: datetime) -> datetime:
    floored = floor_to_6min(t)
    if floored == t.replace(second=0, microsecond=0) and t.second == 0 and t.microsecond == 0:
        return floored
    return floored + timedelta(minutes=6)


def make_6min_targets(start: datetime, end: datetime):
    """生成 [start, end] 范围内所有整 6 分钟目标时刻。"""
    current = ceil_to_6min(start)
    targets = []
    while current <= end:
        targets.append(current)
        current += timedelta(minutes=6)
    return targets


target_times = []
for start_idx, end_idx in segments:
    target_times.extend(make_6min_targets(radar_times[start_idx], radar_times[end_idx]))

target_times = sorted(set(target_times))

target_summary = pd.DataFrame({'target_time': target_times})
print('目标 6 分钟时刻数量:', len(target_times))
target_summary.head(), target_summary.tail()

## 4. 对单个目标时刻做线性插值

对任意目标时刻 `t`，找到前后两个源雷达时刻：

- `t0 <= t <= t1`
- 权重 `w = (t - t0) / (t1 - t0)`
- 插值结果：`arr = arr0 * (1 - w) + arr1 * w`

如果目标时刻刚好等于源文件时刻，则直接复制该源数组。

In [ ]:
def find_bracketing_indices(times, target):
    """找到 target 前后两个源时刻索引。"""
    right = bisect_right(times, target)
    left = right - 1

    if left < 0 or right >= len(times):
        return None

    if times[left] == target:
        return left, left

    return left, right


def interpolate_one_target(target, times, files, max_gap_minutes=30):
    """读取目标时刻前后两帧，并返回插值后的二维雷达反射率数组。"""
    pair = find_bracketing_indices(times, target)
    if pair is None:
        return None

    left, right = pair
    if left == right:
        return np.load(files[left], allow_pickle=False).astype(np.float32, copy=False)

    t0, t1 = times[left], times[right]
    if t1 - t0 > timedelta(minutes=max_gap_minutes):
        return None

    arr0 = np.load(files[left], allow_pickle=False).astype(np.float32, copy=False)
    arr1 = np.load(files[right], allow_pickle=False).astype(np.float32, copy=False)

    weight = (target - t0).total_seconds() / (t1 - t0).total_seconds()
    return (arr0 * (1 - weight) + arr1 * weight).astype(np.float32)


# 先测试一个目标时刻
sample_target = target_times[0]
sample_arr = interpolate_one_target(sample_target, radar_times, radar_files, MAX_GAP_MINUTES)

print('样例目标时刻:', sample_target)
print('shape:', sample_arr.shape)
print('dtype:', sample_arr.dtype)
print('min/max:', float(np.nanmin(sample_arr)), float(np.nanmax(sample_arr)))

## 5. 批量插值并保存

输出文件命名为：`Z9002_YYYYmmddHHMMSS.npy`，例如 `Z9002_20240526001200.npy`。

保存目录：`test_radar_6min/`。

In [ ]:
def output_name_for_time(t: datetime) -> str:
    return f'Z9002_{t:%Y%m%d%H%M%S}.npy'


records = []

for i, target in enumerate(target_times, start=1):
    out_path = OUTPUT_DIR / output_name_for_time(target)

    arr = interpolate_one_target(target, radar_times, radar_files, MAX_GAP_MINUTES)
    if arr is None:
        records.append({'target_time': target, 'output_file': None, 'status': 'skipped'})
        continue

    np.save(out_path, arr)
    records.append({'target_time': target, 'output_file': out_path.name, 'status': 'saved'})

    if i % 50 == 0 or i == len(target_times):
        print(f'{i}/{len(target_times)} done')

interpolation_log = pd.DataFrame(records)
interpolation_log.to_csv(OUTPUT_DIR / 'interpolation_log.csv', index=False)

interpolation_log['status'].value_counts(), interpolation_log.head(), interpolation_log.tail()

## 6. 检查输出结果

In [ ]:
saved_files = sorted(OUTPUT_DIR.glob('Z9002_*.npy'))
print('已保存文件数量:', len(saved_files))

if saved_files:
    check_arr = np.load(saved_files[0], allow_pickle=False)
    print('第一个输出文件:', saved_files[0].name)
    print('shape:', check_arr.shape)
    print('dtype:', check_arr.dtype)
    print('min/max:', float(np.nanmin(check_arr)), float(np.nanmax(check_arr)))

pd.DataFrame({'file': [p.name for p in saved_files[:5]]})

## 7. 可选：配合经纬度读取

插值只改变时间维度，不改变空间网格。输出数组仍然与 `radar_situation.npz` 中的 `lon`、`lat` 一一对应。

In [ ]:
radar_situation = np.load(TEST_DIR / 'radar_situation.npz', allow_pickle=False)
lon = radar_situation['lon']
lat = radar_situation['lat']

print('lon shape:', lon.shape, 'range:', float(lon.min()), float(lon.max()))
print('lat shape:', lat.shape, 'range:', float(lat.min()), float(lat.max()))

assert lon.shape == lat.shape == check_arr.shape